## Connexion SQL

In [0]:
sql_server = dbutils.secrets.get(scope="energy-bi-scope", key="sql-server-name")
sql_database = dbutils.secrets.get(scope="energy-bi-scope", key="sql-database-name")
sql_username = dbutils.secrets.get(scope="energy-bi-scope", key="sql-username")
sql_password = dbutils.secrets.get(scope="energy-bi-scope", key="sql-password")

jdbc_url = f"jdbc:sqlserver://{sql_server}:1433;database={sql_database};encrypt=true;trustServerCertificate=False;hostNameInCertificate=*.database.windows.net;loginTimeout=30"

jdbc_props = {
    "user": sql_username,
    "password": sql_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}

print("Connexion SQL configurée")

## Charger la table Silver

In [0]:
storage_account_name = dbutils.secrets.get(scope="energy-bi-scope", key="blob-storage-account-name")
storage_account_key  = dbutils.secrets.get(scope="energy-bi-scope", key="blob-storage-account-key")
container_name_gold = "gold"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    storage_account_key
)
print("Connexion OK")

In [0]:
from pyspark.sql import functions as F

silver_path = f"wasbs://silver@{storage_account_name}.blob.core.windows.net/energy/"

df_silver = (
    spark.read.option("header", "true").option("inferSchema", "true").csv(silver_path)
)

print(f"Silver lignes : {df_silver.count()}")
df_silver.printSchema()

## Créer DimDate

In [0]:
from pyspark.sql import functions as F

df_dim_date = (
    df_silver.select(F.col("Date"))
    .distinct()
    .withColumnRenamed("Date", "date_key")
    .withColumn("year", F.year("date_key"))
    .withColumn("month", F.month("date_key"))
    .withColumn("month_name", F.date_format("date_key", "MMMM"))
    .withColumn("quarter", F.quarter("date_key"))
    .withColumn("day_of_week", F.dayofweek("date_key"))
    .withColumn("day_name", F.date_format("date_key", "EEEE"))
    .withColumn("is_weekend", (F.dayofweek("date_key").isin([1, 7])).cast("boolean"))
    .orderBy("date_key")
)

df_dim_date.write.format("jdbc").option("url", jdbc_url).option(
    "dbtable", "dbo.DimDate"
).options(**jdbc_props).mode("overwrite").save()

print(f"DimDate : {df_dim_date.count()} lignes")
df_dim_date.show(5)

## Créer DimMeter

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql import functions as F

# Calculer min/max dates depuis Silver
date_bounds = df_silver.agg(
    F.min("Date").alias("date_start"), F.max("Date").alias("date_end")
).collect()[0]

date_start = str(date_bounds["date_start"])
date_end = str(date_bounds["date_end"])

print(f"Date start : {date_start}")
print(f"Date end : {date_end}")

# Construire DimMeter
data = [
    (
        1,
        "Individual Household Sceaux",
        "Residential",
        "Sceaux, France",
        "France",
        1,
        3,
        date_start,
        date_end,
    )
]

schema = StructType(
    [
        StructField("meter_id", IntegerType(), False),
        StructField("meter_name", StringType(), True),
        StructField("meter_type", StringType(), True),
        StructField("location", StringType(), True),
        StructField("country", StringType(), True),
        StructField("sampling_rate_minutes", IntegerType(), True),
        StructField("sub_meter_count", IntegerType(), True),
        StructField("date_start", StringType(), True),
        StructField("date_end", StringType(), True),
    ]
)

df_dim_meter = spark.createDataFrame(data, schema)

df_dim_meter.write.format("jdbc").option("url", jdbc_url).option(
    "dbtable", "dbo.DimMeter"
).options(**jdbc_props).mode("overwrite").save()

print(f"DimMeter : {df_dim_meter.count()} ligne")
df_dim_meter.show(truncate=False)

## Créer FactEnergyGlobal

In [0]:
df_fact_global = (
    df_silver.withColumn(
        "unmetered_wh",
        (F.col("Global_active_power") * 1000 / 60)
        - F.col("Sub_metering_1")
        - F.col("Sub_metering_2")
        - F.col("Sub_metering_3"),
    )
    .groupBy("Date")
    .agg(
        F.sum("Global_active_power").alias("total_active_power_kwh"),
        F.sum("Global_reactive_power").alias("total_reactive_power_kwh"),
        F.avg("Voltage").alias("avg_voltage"),
        F.avg("Global_intensity").alias("avg_intensity"),
        F.sum("unmetered_wh").alias("total_unmetered_wh"),
        F.count("*").alias("record_count"),
        F.count(F.when(F.col("Global_active_power").isNull(), 1)).alias(
            "missing_values_count"
        ),
    )
    .withColumnRenamed("Date", "date_key")
    .withColumn("meter_id", F.lit(1))
    .orderBy("date_key")
)

df_fact_global.write.format("jdbc").option("url", jdbc_url).option(
    "dbtable", "dbo.FactEnergyGlobal"
).options(**jdbc_props).mode("overwrite").save()

print(f"FactEnergyGlobal : {df_fact_global.count()} lignes")
df_fact_global.show(5)

## Créer FactSubMetering

In [0]:
from pyspark.sql.functions import lit

df_sub1 = df_silver.select(
    F.col("Date").alias("date_key"),
    lit(1).alias("meter_id"),
    lit("Sub_Metering_1").alias("sub_meter_name"),
    F.col("Sub_metering_1").alias("energy_wh"),
)
df_sub2 = df_silver.select(
    F.col("Date").alias("date_key"),
    lit(1).alias("meter_id"),
    lit("Sub_Metering_2").alias("sub_meter_name"),
    F.col("Sub_metering_2").alias("energy_wh"),
)
df_sub3 = df_silver.select(
    F.col("Date").alias("date_key"),
    lit(1).alias("meter_id"),
    lit("Sub_Metering_3").alias("sub_meter_name"),
    F.col("Sub_metering_3").alias("energy_wh"),
)

df_fact_sub = (
    df_sub1.union(df_sub2)
    .union(df_sub3)
    .groupBy("date_key", "meter_id", "sub_meter_name")
    .agg(F.sum("energy_wh").alias("total_energy_wh"))
    .orderBy("date_key", "sub_meter_name")
)

df_fact_sub.write.format("jdbc").option("url", jdbc_url).option(
    "dbtable", "dbo.FactSubMetering"
).options(**jdbc_props).mode("overwrite").save()

print(f"FactSubMetering : {df_fact_sub.count()} lignes")
df_fact_sub.show(5)

##  Vérification finale

In [0]:
tables = ["DimDate", "DimMeter", "FactEnergyGlobal", "FactSubMetering"]

for table in tables:
    df_check = (
        spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", f"dbo.{table}")
        .options(**jdbc_props)
        .load()
    )
    print(f"dbo.{table} > {df_check.count()} lignes")